In [1]:
import re
import torch
import datasets
import numpy as np
import transformers
import shap

from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

### Load Reward Model

In [2]:
reward_model_path = "cleanrl/EleutherAI_pythia-1b-deduped__reward__tldr"

reward_model = AutoModelForSequenceClassification.from_pretrained(
    reward_model_path,
    num_labels=1,
).to("cuda")

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [3]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-1b-deduped")

In [4]:
tokenizer.add_special_tokens({"pad_token": "[PAD]"})
pad_token_id = tokenizer.convert_tokens_to_ids("[PAD]")
reward_model.config.pad_token_id = pad_token_id

In [5]:
tokenizer

GPTNeoXTokenizer(name_or_path='EleutherAI/pythia-1b-deduped', vocab_size=50254, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '[PAD]'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|padding|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50254: AddedToken("                        ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50255: AddedToken("                       ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50256: AddedToken("                      ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50257: AddedToken("                     ", rstrip=False, lstrip=False,

In [6]:
import re

def parse_sentence(paragraph, return_offsets_mapping=True):
    # ; → single
    # ?+ !+ → repeated punctuation stays together
    # .+ → repeated dots stay together, but not if inside numbers (2.5)
    pattern = r";+|\?+|!+|(?<!\d)\.+(?!\d)"

    spans = []
    offset_mapping = []
    start = 0

    for match in re.finditer(pattern, paragraph):
        end = match.end()
        span = paragraph[start:end]
        if span:
            spans.append(span)
            offset_mapping.append((start, end))
        start = end

    if start < len(paragraph):
        spans.append(paragraph[start:])
        offset_mapping.append((start, len(paragraph)))

    if return_offsets_mapping:
        return {"input_ids": spans, "offset_mapping": offset_mapping}
    else:
        return {"input_ids": spans}

In [7]:
def first_true_indices(bools, dtype=torch.long):
    """
    Takes an N-dimensional bool tensor and returns an (N-1)-dimensional tensor of integers giving
    the position of the first True in each "row".

    Returns the length of the rows (bools.size(-1)) if no element is True in a given row.
    """
    row_len = bools.size(-1)
    zero_or_index = row_len * (~bools).type(dtype) + torch.arange(row_len, dtype=dtype, device=bools.device)
    return torch.min(zero_or_index, dim=-1).values

def get_shap_rewards(model, query_str, response_str, tokenizer, masker=None):
    def f(x):
        partial_sentences = []
        for _x in x:
            if len(_x) > 0 and _x[0] == " ":
                concatenated = query_str + _x
            else:
                concatenated = query_str + " " + _x
            concatenated += tokenizer.eos_token
            partial_sentences.append(concatenated)

        # print(partial_sentences)
        query_responses = tokenizer(
            partial_sentences,
            padding="longest",
            padding_side="right",
            return_tensors="pt",
        ).to("cuda")["input_ids"]

        attention_mask = query_responses != tokenizer.pad_token_id
        position_ids = attention_mask.cumsum(1) - attention_mask.long()  # exclusive cumsum
        lm_backbone = getattr(model, model.base_model_prefix)
        input_ids = torch.masked_fill(query_responses, ~attention_mask, 0)
        output = lm_backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            return_dict=True,
            output_hidden_states=True,
            use_cache=False,  # otherwise mistral-based RM would error out
        )
        reward_logits = model.score(output.hidden_states[-1])
        sequence_lengths = first_true_indices(input_ids == tokenizer.pad_token_id) - 1

        return reward_logits[
            torch.arange(reward_logits.size(0), device=reward_logits.device),
            sequence_lengths,
        ].squeeze(-1)

    masker = tokenizer if not masker else masker
    explainer = shap.Explainer(f, masker, algorithm="auto")
    shap_values = explainer([response_str])

    return shap_values

In [8]:
# test_examples = [
#     "I still have contact with an old ex's friends but can't stand to see or talk to him. His friends are really nice, so how do I tell them I possibly want to unfriend them on Facebook because of him?",
#     "Progress is still happening, even when you think it might not be! Don't get discouraged, even if your journey seems to be going slowly. Don't give up, warriors.",
#     "My skin is scarred badly; what could I do/say about it that would gross my future partner out the least? What's your experience with body image issues?",
#     "$14k in student debt (all <5%) and need to save more for down payment on a ~$300k house. How to allocate $5200/mo between the two?",
#     "GF is a meanie-bo-beanie when I'm nice, and an absolute doll when I'm uninterested. Sex is bomb and she's the hottest I've ever dated. What do?",
#     "how do I deny sex with my boyfriend of 2.5 years without him feeling like I don't want to please him?",
#     "HOW do I introduce new people? HOW do I introduce new dogs? WHAT do I do about 4th of July??"
# ]

from datasets import load_dataset

# Load the full dataset with default splits
dataset = load_dataset("trl-lib/tldr")

# Example: access the "train" split (if present)
if "train" in dataset:
    train_ds = dataset["train"]
else:
    # If it's not split into train/valid/test, use the default
    train_ds = dataset

# Show the first example
print(train_ds[0])

{'prompt': "SUBREDDIT: r/relationships\n\nTITLE: I (f/22) have to figure out if I want to still know these girls or not and would hate to sound insulting\n\nPOST: Not sure if this belongs here but it's worth a try. \n\nBackstory:\nWhen I (f/22) went through my first real breakup 2 years ago because he needed space after a year of dating roand  it effected me more than I thought. It was a horrible time in my life due to living with my mother and finally having the chance to cut her out of my life. I can admit because of it was an emotional wreck and this guy was stable and didn't know how to deal with me. We ended by him avoiding for a month or so after going to a festival with my friends. When I think back I wish he just ended. So after he ended it added my depression I suffered but my friends helped me through it and I got rid of everything from him along with cutting contact. \n\nNow: Its been almost 3 years now and I've gotten better after counselling and mild anti depressants. My m

In [9]:
index = 8

shap_outputs = get_shap_rewards(
    reward_model,
    train_ds[index]["prompt"],
    train_ds[index]["completion"],
    tokenizer,
    masker=shap.maskers.Text(parse_sentence, mask_token=" ", collapse_mask_token=True)
)

In [10]:
shap.plots.text(shap_outputs)

In [11]:
shap_outputs.values.sum() + shap_outputs.base_values

array([2.7408669])

In [12]:
shap_outputs

.values =
array([[2.10883602, 0.25172651, 1.91182414, 1.55387866]])

.base_values =
array([-3.08539844])

.data =
(array([' I think I might like two girls at once.', " Don't want to;",
       ' just want to be head over heels for one.',
       ' Do I do anything before college?'], dtype=object),)

In [13]:
train_ds[5]["completion"]

' I asked guy I have been dating for a month if everything was okay and now I regret it because I think it makes me look pushy /clingy.'